# College Admission Cutoff Analysis

## Overview

This subsection investigates the **admission process for junior colleges (JCs)** in Telangana, India, focusing on understanding cutoff patterns and allocation mechanisms. The core question: *Can we infer admission cutoffs from observed allocations, and do they follow predictable patterns based on student preferences and entrance marks?*

## Research Context

### The Admission System
Students apply to junior colleges by:
1. Taking an entrance examination (`marks`)
2. Submitting ranked preferences (up to 20 colleges)
3. Being allocated to colleges through a centralized algorithm

The system theoretically implements:
- **Category-based reservations**: SC, ST, BC, OC/EBC quotas
- **Preference-based allocation**: Higher preferences should be considered first
- **Merit-based selection**: Higher marks should improve admission chances


Initial analysis revealed **puzzling patterns** that contradict expected admission logic:
- Students with lower marks getting admitted while higher-scoring applicants are rejected
- No clear monotonic relationship between marks and admission success
- Substantial overlap between "admitted" and "failed applicant" score distributions

## Methodology

### Approach 1: Theoretical Cutoff Estimation (FAILED)

**Concept**: Find the highest-scoring student who wanted a college but didn't get in → this should be the cutoff.

**Why it failed**:
- Produced unrealistic cutoffs (often 100+ marks)
- Failed students often had **higher marks** than admitted students
- No clear separation between admitted/rejected score distributions

### Approach 2: Empirical Distribution Analysis

**Concept**: Instead of assuming cutoffs exist, visualize actual admission patterns.

**Implementation**:
- For each college, separate students into:
  - **Admitted**: Those allocated to this college
  - **Failed**: Those who ranked it but got a worse preference
- Compare score distributions using histograms
- Stratify by:
  - Social category (SC/ST, BC, OC/EBC)
  - Preference rank (1st choice vs 2nd choice vs 3rd choice, etc.)

## Key Findings

### 1. No Clear Cutoff Structure
```
Example: College 506404_cec
Failed applicants: Max mark = 74.75
Admitted students: Max mark = 60.00

Interpretation: Someone who scored 74.75 didn't get in, 
but someone who scored 60.00 did.
```

This pattern appears **consistently across colleges**, not as isolated anomalies.

### 2. Category-Based Patterns Are Weak
Breaking down by social category (SC/ST, BC, OC/EBC) does NOT clarify the picture:
- Within SC/ST: Still massive overlap between admitted/failed
- Within BC: Same issue persists
- Within OC/EBC: Too few students to draw conclusions

### 3. Preference Rank Shows Irregular Patterns
```
Example: College 501502_bpc
Pref Rank 1: Min = 16.0, Max = 68.0  (40 students)
Pref Rank 2: Min = 18.75, Max = 59.0 (11 students)
Pref Rank 5: Min = 25.25, Max = 77.5 (7 students)
Pref Rank 6: Min = 77.75, Max = 77.75 (1 student)
```

**Observations**:
- Rank 6 has HIGHER marks than Rank 1-5
- No monotonic decrease in marks with lower preference ranks
- Huge variation within each preference rank

### 4. Most Students Admitted at Top 3 Preferences
The vast majority of admitted students used preferences 1-3, but even here, mark ranges are enormous.

## Hypotheses for Unexpected Patterns

### 1. **Preference Data Mismatch**
The preference list in the dataset may not reflect the actual list used by the allocation algorithm:
- Data entry errors
- Systematic shifting of preference ranks
- Missing preferences or duplicate entries
- Preferences edited after initial submission

### 2. **Complex Quota System**
Horizontal + vertical reservations interact unpredictably:
- **Vertical categories**: SC, ST, BC, OC/EBC
- **Horizontal reservations**: Gender, minority, disability, rural/urban
- **Seat conversions**: Unfilled category seats may convert to other categories
- **Subcategory quotas**: Within SC, there may be SC-Christian, SC-Buddhist, etc.

Example scenario:
```
College has 10 seats:
- 5 SC seats (3 girls, 2 boys)
- 3 BC seats (1 minority)
- 2 OC seats

A high-scoring SC boy may lose to a lower-scoring SC girl 
if all SC-boy seats are filled.
```

### 3. **Non-Rank-Based Allocation**
The algorithm may NOT process preferences strictly in order:
- **Batch allocation**: All Rank 1 preferences processed, then Rank 2, etc.
- **Tie-breakers**: Age, distance from college, date of application
- **Zonal restrictions**: Some colleges prioritize local students
- **Stream prerequisites**: Previous academic stream requirements

### 4. **Hidden Eligibility Criteria**
Students may appear eligible in data but fail internal checks:
- Language medium requirements (English vs Telugu)
- School type restrictions (government vs private)
- Zonal/regional constraints
- Domicile requirements
- Stream continuity rules

### 5. **Multi-Round Allocation**
If allocation happens in multiple rounds:
- Early rounds may consume seats unpredictably
- Later students face different availability
- Seat matrices change between rounds
- Our dataset may capture only the final state

### 6. **Data Quality Issues**
The `allotted_JC_code` field shows anomalies:
- Some codes don't match any preference list
- Format inconsistencies (spacing, suffixes)
- Missing or null allocations
- Potential data collection timing issues

## Data Structure

### Input Dataset
- **File**: `1.5kcolleges_student&collegedetails_v2_limitedvars.dta`
- **Total students**: 39,435
- **Admitted to JCs**: 6,414 (16.3%)
- **Not admitted**: 33,021 (83.7%)

### Key Variables
```
marks                  - Entrance exam score (range: -17 to 128.75)
allotted_JC_code       - Allocated college code (6-digit + stream suffix)
general_pref1..20      - Student's preference rankings
social_category        - SC, ST, BC, OC/EBC
is_category            - Admission category type
englishmarks_12        - Grade 12 English outcome
marks12th              - Grade 12 total marks
```

### Sample Statistics
```
Entrance marks:
  Mean: 37.1, SD: 18.1
  Range: [-5, 108.75]

12th English marks:
  Mean: 77.5, SD: 14.6
  Range: [5, 98]

Correlation (entrance vs English): r = 0.407
Correlation (entrance vs 12th total): r = 0.528
```

## Code Structure

### Phase 1: Data Exploration
```
1. Load and clean data
2. Visualize score distributions
3. Count admissions by status
4. Map college codes to streams
5. Identify top colleges by enrollment
```

### Phase 2: Cutoff Estimation (Conservative)
```
1. Normalize college codes and preferences
2. Build preference rank mapping for each student
3. For each college:
   - Identify students who listed it
   - Determine if they got a worse preference
   - Find highest mark among failed applicants
4. Plot estimated cutoffs
```

**Key finding**: All cutoffs unrealistically high, method abandoned.

### Phase 3: Distribution Analysis
```
1. Define helper functions:
   - get_failed_marks_for_jc(jc_code)
   - get_admitted_marks_for_jc(jc_code)
2. For selected colleges:
   - Generate side-by-side histograms
   - Use common X-axis for comparison
3. Stratify by social category
4. Search for patterns
```

### Phase 4: Preference-Rank Decomposition
```
1. For each college:
   For each preference rank (1-20):
     - Count admitted students at this rank
     - Find min/max marks at this rank
     - Tabulate results
2. Look for monotonic patterns
3. Identify irregularities
```

## Validation Checks Performed

### 1. Preference Mapping Validation
Random sample of 10 students: ✓ All preference ranks correctly mapped

### 2. Logic Validation
For 10 random applicants to sample college:
- Verified preference rank extraction
- Checked allotted college lookup
- Confirmed "failed applicant" classification logic

### 3. Category Filtering
Verified SC/ST/BC/OC groupings produce expected counts:
- SC_ST: 19,057
- BC: 19,043
- OC_EBC: 1,293
- OTHER: 42

### 4. Admitted Students Inspection
For suspicious cases (e.g., high-mark failed + low-mark admitted), manually inspected full records to confirm data accuracy.

## Visualizations Generated

1. **Marks distribution by admission status** (boxplot)
2. **Correlation heatmap** of mark-like variables
3. **Top 20 college enrollment** (bar chart)
4. **Top 25 variables by missingness**
5. **Entrance marks histogram**
6. **12th total marks histogram**
7. **12th English marks histogram**
8. **Failed vs Admitted distributions** (side-by-side histograms)
9. **Category-stratified distributions** (3×2 grid per college)
10. **Estimated cutoffs** (bar chart) — unrealistic results

## Interpretation & Implications

### What We Know
1. **Admission is not purely merit-based**: Higher marks don't guarantee admission
2. **Preferences matter, but unpredictably**: Even top preferences show huge mark ranges
3. **Categories introduce complexity**: But don't fully explain the patterns
4. **The allocation algorithm is opaque**: Actual rules differ from apparent logic

### What We Don't Know
1. **True allocation algorithm**: Exact tiebreaker rules, batch processing, seat conversions
2. **Hidden eligibility filters**: Zonal restrictions, medium requirements, etc.
3. **Data completeness**: Are all preferences recorded? Is timing correct?
4. **Sub-quota structure**: Detailed breakdown of seat reservations

### Implications for Students
- **Don't rely on simple cutoff estimates**: They don't exist in a meaningful way
- **Preference ordering matters**: But how it matters is unclear
- **Category advantage varies**: Not uniformly predictable across colleges
- **Lower marks ≠ no chance**: System complexity means opportunities exist at many score levels

### Implications for Policy
- **Transparency needed**: Current system is a "black box" to applicants
- **Predictability matters**: Students can't make informed choices without clear rules
- **Data quality issues**: Official records may not accurately capture the allocation process
- **Algorithm documentation**: Rules should be published and verifiable

## Limitations

1. **Single cohort**: Analysis is cross-sectional, not longitudinal
2. **No algorithm access**: We're reverse-engineering from outcomes
3. **Incomplete data**: Missing variables (zonal info, exact quotas, etc.)
4. **Conservative assumptions**: We may be overcounting "failed applicants"
5. **Static analysis**: Doesn't capture multi-round allocation dynamics

## Future Directions

### If Algorithm Access Were Available
1. Simulate allocations under different assumptions
2. Test sensitivity to tie-breaker rules
3. Validate cutoff estimation methods
4. Build predictive admission models

### With Additional Data
1. Zonal/regional restrictions
2. Exact seat matrices by category and subcategory
3. Round-by-round allocation states
4. Complete eligibility criteria
5. Multi-year historical data

### Alternative Analyses
1. **Regression discontinuity**: Look for sharp changes near score thresholds
2. **Propensity score matching**: Estimate treatment effects of college attendance
3. **Network analysis**: How do preference patterns create competition?
4. **Machine learning**: Can we predict admission from available features?

## Technical Notes

### Data Cleaning Steps
```python
1. Normalize college codes:
   - Convert to lowercase
   - Strip whitespace
   - Remove ".0" suffixes
   - Replace placeholders (nan, none, 0) with NaN

2. Build preference rank mapping:
   - For each student, create dict: {college_code: rank}
   - Handle missing preferences gracefully

3. Standardize categories:
   - Map social_category to SC_ST, BC, OC_EBC groups
   - Filter by is_category == "general"
```

### Conservative Counting Rules
When determining "failed applicants":
- **SKIP** students with no allocation (they may have withdrawn)
- **SKIP** if allotted college not in student's preferences (data mismatch)
- **COUNT** only if allotted rank > target college rank (definite failure)

This produces **lower bounds** on cutoffs, yet they're still unrealistic.

## Dependencies

```python
pandas>=2.0
numpy>=2.0
matplotlib>=3.7
```

## Files Generated

None — this is an exploratory analysis notebook. All results are displayed inline.

## Conclusion

The college admission process in this dataset **does not follow simple cutoff-based logic**. Multiple interacting factors—complex quota systems, hidden eligibility rules, algorithmic details, and potential data quality issues—make it impossible to infer clean cutoffs from observed allocations.

The analysis successfully demonstrates **why naive cutoff estimation fails**, and provides a framework for understanding the sources of allocation complexity. However, without access to the actual allocation algorithm or additional administrative data, we cannot definitively explain the observed patterns.

This work highlights the **need for transparency** in educational allocation systems, both for fairness to applicants and for enabling rigorous policy evaluation.

---

**Analysis Date**: January 2025  
**Codebase**: Exploratory Jupyter notebook  
**Status**: Hypothesis generation phase — patterns identified, explanations remain uncertain